# TP2 — Encodeur Transformer hiérarchique en monde ouvert (Colab, GPU T4)

**Rôle de ce notebook.** Il **calcule** ce qui exige un GPU : pour chacune des **5 rotations** de classes masquées, il entraîne un encodeur (DistilBERT) avec **deux têtes** (famille et classe fine) sur les seules classes **connues**, puis produit, pour les logs de validation, les **sorties brutes** (logits des deux têtes) et les probabilités moyennes de **Monte-Carlo dropout** (10 passages avec dropout actif).

**Zone de vérité.** Le notebook principal local `TP2_logs_hierarchique_monde_ouvert.ipynb` contient les explications, les scores d'incertitude (probabilité maximale, énergie, MC dropout...), les analyses et graphiques. Ce notebook-ci ne fait que calculer ; il n'utilise **jamais** le jeu de test (absent du fichier de données).

**Mode d'emploi.** (1) Exécution > Modifier le type d'exécution > **GPU T4**. (2) Cellule 2 : déposer **uniquement** `encoder_data.csv.gz` (dossier `TP/TP2/colab/`) ; la taxonomie est écrite par la cellule suivante. (3) *Tout exécuter* (environ 1 h ; la liste `RUNS` peut être découpée avec `START`/`END`). (4) La dernière cellule télécharge `tp2_colab_out.zip` : à décompresser dans `TP/TP2/colab/out/`.

**Plan des runs (décidé avant les résultats).** `distilbert-base-uncased` sur les 5 rotations (avec MC dropout) ; `distilbert-base-cased` sur les 5 rotations (sans MC dropout, pour comparer casse et non-casse) ; contrôle de fenêtre 512 contre 256 jetons sur la rotation 1. Une seule graine (42) : dit comme limite. Réglages usuels, non optimisés : 2 époques, taux d'apprentissage 3e-5, lot 32.

In [ ]:
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN : changer le type d\'exécution')

In [ ]:
import os
from google.colab import files
if not os.path.exists('encoder_data.csv.gz'):
    files.upload()   # choisir UNIQUEMENT encoder_data.csv.gz
assert os.path.exists('encoder_data.csv.gz'), "Fichier absent : relancer cette cellule et choisir encoder_data.csv.gz"
print('Fichiers présents :', sorted(os.listdir('.')))

In [ ]:
%%writefile taxonomy_and_rotations_v1.json
{
 "taxonomy": {
  "authentication": [
   "authentication-failed",
   "authentication-success"
  ],
  "identity_session": [
   "user-creation",
   "user-deletion",
   "user-logout",
   "user-session-open"
  ],
  "network": [
   "connection-closed",
   "connection-failed",
   "connection-opened",
   "network-filtered",
   "network-traffic"
  ],
  "web": [
   "http-request-failure",
   "http-request-success"
  ],
  "file": [
   "directory-changed",
   "directory-created",
   "directory-deleted",
   "file-action-failure",
   "file-deleted",
   "file-modification",
   "file-read",
   "file-write"
  ],
  "process": [
   "process-ended",
   "process-error",
   "process-info",
   "process-shutdown",
   "process-started"
  ],
  "system": [
   "database-operation",
   "hardware-monitoring",
   "system-configuration-changed"
  ],
  "security_alert": [
   "ids-alert"
  ]
 },
 "rotations": [
  [
   "directory-created",
   "http-request-success",
   "process-info",
   "process-started",
   "system-configuration-changed",
   "user-session-open"
  ],
  [
   "database-operation",
   "file-deleted",
   "network-traffic",
   "process-ended",
   "process-error",
   "user-creation"
  ],
  [
   "authentication-failed",
   "connection-failed",
   "directory-changed",
   "file-action-failure",
   "http-request-failure",
   "network-filtered"
  ],
  [
   "file-modification",
   "file-read",
   "hardware-monitoring",
   "ids-alert",
   "process-shutdown",
   "user-logout"
  ],
  [
   "authentication-success",
   "connection-closed",
   "connection-opened",
   "directory-deleted",
   "file-write",
   "user-deletion"
  ]
 ],
 "seed": 42,
 "tries": 1,
 "classes_paper_order": [
  "authentication-failed",
  "authentication-success",
  "connection-closed",
  "connection-failed",
  "connection-opened",
  "database-operation",
  "directory-changed",
  "directory-created",
  "directory-deleted",
  "file-action-failure",
  "file-deleted",
  "file-modification",
  "file-read",
  "file-write",
  "hardware-monitoring",
  "http-request-failure",
  "http-request-success",
  "ids-alert",
  "network-filtered",
  "network-traffic",
  "process-ended",
  "process-error",
  "process-info",
  "process-shutdown",
  "process-started",
  "system-configuration-changed",
  "user-creation",
  "user-deletion",
  "user-logout",
  "user-session-open"
 ]
}

In [ ]:
%%writefile tp2_encoder.py
"""Encodeur Transformer hiérarchique pour logs en monde ouvert (TP2, étape 6). Calcul sur GPU (Colab ou Kaggle).

Une base (DistilBERT / BERT) + deux têtes linéaires : famille (F sorties) et classe fine (K sorties, classes CONNUES seulement).
Perte = entropie croisée famille + entropie croisée classe fine. À l'inférence : logits des deux têtes, et Monte-Carlo dropout
(T passages avec dropout actif) pour estimer l'incertitude. Les scores d'incertitude sont calculés ensuite, en local.
"""
import json
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup


def pick_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class HierEncoder(nn.Module):
    def __init__(self, name, n_fine, n_fam, dropout=0.1):
        super().__init__()
        self.base = AutoModel.from_pretrained(name)
        h = self.base.config.hidden_size
        self.drop = nn.Dropout(dropout); self.fam = nn.Linear(h, n_fam); self.fine = nn.Linear(h, n_fine)

    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float(); pooled = (out * m).sum(1) / m.sum(1)      # moyenne des jetons, remplissage exclu
        z = self.drop(pooled)
        return self.fam(z), self.fine(z)


def _batch(ids, idx, pad_id, device):
    L = max(len(ids[j]) for j in idx)
    x = torch.full((len(idx), L), pad_id, dtype=torch.long); m = torch.zeros((len(idx), L), dtype=torch.long)
    for r, j in enumerate(idx):
        x[r, :len(ids[j])] = torch.tensor(ids[j]); m[r, :len(ids[j])] = 1
    return x.to(device), m.to(device)


def train_encoder(name, texts, y_fine, y_fam, n_fine, n_fam, *, max_len=256, epochs=2, lr=3e-5, batch_size=32, seed=42, device=None, warmup=0.06):
    device = device or pick_device(); set_seed(seed)
    tok = AutoTokenizer.from_pretrained(name); model = HierEncoder(name, n_fine, n_fam).to(device)
    ids = tok(list(texts), truncation=True, max_length=max_len)["input_ids"]
    yf = torch.tensor(np.asarray(y_fine), dtype=torch.long); yfa = torch.tensor(np.asarray(y_fam), dtype=torch.long)
    total = math.ceil(len(ids) / batch_size) * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01); sched = get_linear_schedule_with_warmup(opt, int(warmup * total), total)
    ce = nn.CrossEntropyLoss(); rng = np.random.default_rng(seed); step = 0; t0 = time.perf_counter(); hist = []
    for ep in range(epochs):
        model.train(); perm = rng.permutation(len(ids)); run = 0.0
        for s in range(0, len(perm), batch_size):
            idx = perm[s:s + batch_size]; x, m = _batch(ids, idx, tok.pad_token_id, device)
            lf, lc = model(x, m); loss = ce(lf, yfa[idx].to(device)) + ce(lc, yf[idx].to(device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            step += 1; run += float(loss.item())
            if step % 100 == 0: hist.append({"step": step, "loss": run / 100, "min": round((time.perf_counter() - t0) / 60, 2)}); run = 0.0
    info = {"model": name, "max_len": max_len, "epochs": epochs, "lr": lr, "batch_size": batch_size, "seed": seed, "device": device, "steps": step,
            "train_minutes": round((time.perf_counter() - t0) / 60, 2), "n_train": len(ids), "history": hist}
    return model, tok, info


@torch.no_grad()
def predict(model, tok, texts, *, max_len=256, batch_size=128, mc=10, device=None):
    device = device or pick_device(); model.to(device)
    ids = tok(list(texts), truncation=True, max_length=max_len)["input_ids"]; order = np.argsort([len(x) for x in ids])
    n = len(ids); nf = model.fine.out_features; nfa = model.fam.out_features
    out = {"logits_fine": np.zeros((n, nf), np.float32), "logits_fam": np.zeros((n, nfa), np.float32), "mc_fine": np.zeros((n, nf), np.float32), "mc_fam": np.zeros((n, nfa), np.float32)}
    for s in range(0, n, batch_size):
        idx = order[s:s + batch_size]; x, m = _batch(ids, idx, tok.pad_token_id, device)
        model.eval(); lf, lc = model(x, m); out["logits_fam"][idx] = lf.float().cpu().numpy(); out["logits_fine"][idx] = lc.float().cpu().numpy()
        if mc:
            model.train()                                                                    # dropout actif : T passages stochastiques
            pf = torch.zeros_like(lf); pc = torch.zeros_like(lc)
            for _ in range(mc):
                a, b = model(x, m); pf += torch.softmax(a.float(), -1); pc += torch.softmax(b.float(), -1)
            out["mc_fam"][idx] = (pf / mc).cpu().numpy(); out["mc_fine"][idx] = (pc / mc).cpu().numpy(); model.eval()
    return out

In [ ]:
import json, time, os, numpy as np, pandas as pd, torch
import tp2_encoder as E
df = pd.read_csv('encoder_data.csv.gz'); tax = json.load(open('taxonomy_and_rotations_v1.json'))
assert set(df.split) == {'train', 'val'}
tr = df[df.split == 'train'].reset_index(drop=True); va = df[df.split == 'val'].reset_index(drop=True)
FAMS = sorted(tax['taxonomy']); PAPER = tax['classes_paper_order']; ROT = tax['rotations']
print(len(tr), 'logs d\'entraînement |', len(va), 'logs de validation |', len(FAMS), 'familles |', len(PAPER), 'classes')
os.makedirs('/content/out', exist_ok=True)

In [ ]:
RUNS = [('distilbert-base-uncased', r, 256, 10) for r in range(5)] + [('distilbert-base-cased', r, 256, 0) for r in range(5)] + [('distilbert-base-uncased', 0, 512, 0)]
START, END = 0, len(RUNS)          # pour reprendre ou découper : par exemple START, END = 0, 5
print(len(RUNS), 'runs prévus')

In [ ]:
for k in range(START, END):
    name, rot, max_len, mc = RUNS[k]; tag = f"{name.split('-')[-1]}_rot{rot + 1}_len{max_len}_s42"
    if os.path.exists(f'/content/out/{tag}.npz'): print('déjà fait :', tag); continue
    hidden = set(ROT[rot]); known = [c for c in PAPER if c not in hidden]; fams_known = sorted({c2 for c in known for c2 in [next(f for f, cs in tax['taxonomy'].items() if c in cs)]})
    c2i = {c: i for i, c in enumerate(known)}; f2i = {f: i for i, f in enumerate(FAMS)}
    t_known = tr[tr.category.isin(known)].reset_index(drop=True)
    model, tok, info = E.train_encoder(name, t_known.text.values, t_known.category.map(c2i).values, t_known.family.map(f2i).values, len(known), len(FAMS), max_len=max_len, epochs=2, lr=3e-5, batch_size=32, seed=42)
    out = E.predict(model, tok, va.text.values, max_len=max_len, mc=mc)
    np.savez_compressed(f'/content/out/{tag}.npz', **out)
    json.dump({'tag': tag, 'known': known, 'families': FAMS, 'hidden': sorted(hidden), 'info': info, 'mc': mc}, open(f'/content/out/{tag}.json', 'w'), indent=1)
    acc = (out['logits_fine'].argmax(1) == va.category.map(lambda c: c2i.get(c, -1)).values)[va.category.isin(known)].mean()
    print(f"{tag}: entraînement {info['train_minutes']} min, exactitude fermée sur les connus (validation) {acc:.3f}")
    del model; torch.cuda.empty_cache()

In [ ]:
!cd /content && zip -qr tp2_colab_out.zip out
from google.colab import files
files.download('/content/tp2_colab_out.zip')